# 🚢 تجارت‌یار — اجرای سامانه روی Google Colab

این دفترچه سامانه **تجارت‌یار** (React + Express + Vite) را روی Colab نصب، build و اجرا می‌کند و یک **لینک عمومی موقت** می‌دهد.

**هر سلول را به ترتیب با `Ctrl+Enter` اجرا کنید** (یا از منوی `Runtime → Run all`):
1. نصب Node.js ۲۰
2. دریافت کد از GitHub
3. نصب وابستگی‌ها
4. build پروژه
5. اجرای سرور (UI + API روی پورت ۳۰۰۰)
6. راه‌اندازی تونل عمومی (cloudflared)
7. دریافت لینک دسترسی

> همه‌ی سلول‌های کد با `%%bash` شروع می‌شوند و به‌صورت خودکار به‌عنوان اسکریپت شل اجرا می‌شوند؛ نیازی به تغییر چیزی نیست.
> لینک نهایی در خروجی **سلول آخر** چاپ می‌شود.

In [ ]:
%%bash
curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs && echo "Node: $(node -v) / npm: $(npm -v)"

In [ ]:
%%bash
cd /content && rm -rf Tejaratyarr && git clone --depth 1 --branch arena/01a04f8e-tejaratyarr https://github.com/Setayesh-Jafari/Tejaratyarr.git

In [ ]:
%%bash
cd /content/Tejaratyarr && npm install

In [ ]:
%%bash
cd /content/Tejaratyarr && npm run build

In [ ]:
%%bash
cd /content/Tejaratyarr && setsid nohup node dist/server.cjs > /content/server.log 2>&1 < /dev/null &

In [ ]:
%%bash
sleep 5; curl -s http://localhost:3000/api/health; echo

In [ ]:
%%bash
cd /content/Tejaratyarr && (test -f cloudflared || wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared) && chmod +x cloudflared && echo cloudflared-ready

In [ ]:
%%bash
cd /content/Tejaratyarr && setsid nohup ./cloudflared tunnel --url http://localhost:3000 > /content/cloudflared.log 2>&1 < /dev/null &

In [ ]:
%%bash
for i in $(seq 1 60); do U=$(grep -oE 'https://[a-z0-9-]+\.trycloudflare\.com' /content/cloudflared.log | head -1); if [ -n "$U" ]; then echo "LINK: $U"; exit 0; fi; sleep 2; done; echo 'no link yet — cloudflared log:'; tail -20 /content/cloudflared.log

## 📌 نکات

- لینک `trycloudflare` **موقت** است و تا زمانی که Colab روشن بماند کار می‌کند.
- اگر لینک چاپ نشد: سلول تونل (شماره ۸) را دوباره اجرا کنید، سپس سلول آخر (شماره ۹) را اجرا کنید.
- داده‌ها در Colab موقتی است؛ برای استفاده‌ی واقعی روی سرور خودتان اجرا کنید.
- فعال‌سازی هوش مصنوعی Gemini: قبل از اجرای سرور، `GEMINI_API_KEY` را در سلول ۵ تنظیم کنید.
- اجرای محلی: `npm install` سپس `npm run dev` (پورت ۳۰۰۰).
- اجرای تولید: `npm run build` سپس `NODE_ENV=production node dist/server.cjs`.

## 🛠 اگر سلولی خطای `SyntaxError` داد
یعنی سلول به‌جای bash به‌صورت پایتون اجرا شده است. مطمئن شوید خط اول سلول `%%bash` باشد (در این دفترچه خودکار است).